In [1]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']

pipeline = [
    # 1. Sicherstellen, dass die Felder existieren
    {
        "$match": {
            "Com autónoma": "Canarias",
            "Temperature high": { "$exists": True },
            "Temperature low": { "$exists": True },
            "rainfall mm": {"$exists": True}
        }
    },

    # 2. Felder in Arrays transformieren
    {
        "$project": {
            "Ciudad": 1,
            "Isla": 1,
            "Población": 1,
            "Densidad": 1,
            "Altitud": 1,
            "high_arr": { "$objectToArray": "$Temperature high" },
            "low_arr": { "$objectToArray": "$Temperature low" },
            "rain_arr": { "$objectToArray": { "$ifNull": ["$rainfall mm", {}] } }
            
        }
    },

    # 3. Berechnungen (Durchschnitte, Min, Max)
    {
        "$addFields": {
            "avg_high": { "$avg": "$high_arr.v" },
            "avg_low": { "$avg": "$low_arr.v" },
            "max_temp": { "$max": "$high_arr.v" },
            "min_temp": { "$min": "$low_arr.v" },
            "avg_rain": { "$avg": "$rain_arr.v" },
            "water_arr": { "$objectToArray": { "$ifNull": ["$water", {}] } }
        }
    },

    # 4. Spanne und Wasser-Durchschnitt
    {
        "$addFields": {
            "avg_water": { "$avg": "$water_arr.v" },
            "amplitude": { "$subtract": ["$max_temp", "$min_temp"] }
        }
    },
    { "$sort": { "avg_high": -1 } }
]

results = collection.aggregate(pipeline)

header = f"{'Ciudad':<30} | {'Isla':<25}  | {'Población':<10} | {'Densidad': <11} | {'Altitud':<5} |  {'High':<6} | {'Low':<6} | {'Spanne':<7} | {'Regen':<7} "
print(header)
print("-" * len(header))

for res in results:
    ciudad = res.get('Ciudad', 'Unbekannt')
    Isla = res.get('Isla', 'Unbekannt')
    Población = res.get('Población', 'Unbekannt')
    Densidad =  res.get('Densidad', 'Unbekannt')
    Altitud =  res.get('Altitud', 'Unbekannt')
    
    # Sicherer Fallback: Wenn ein Wert None ist, wird 0 verwendet
    a_high = round(res.get('avg_high') or 0, 1)
    a_low  = round(res.get('avg_low') or 0, 1)
    amp    = round(res.get('amplitude') or 0, 1)
    rain   = round(res.get('avg_rain') or 0, 1)
    
    w_val  = res.get('avg_water')
    water_str = f"{round(w_val, 1):>6}" if w_val is not None else "  ---  "

    print(f"{ciudad:<30} |  {Isla:<25} |  {Población:>10} | {Densidad:>11} | {Altitud:>5} | {a_high:>6} | {a_low:>6} | {amp:>7} | {rain:>7}")

Ciudad                         | Isla                       | Población  | Densidad    | Altitud |  High   | Low    | Spanne  | Regen   
----------------------------------------------------------------------------------------------------------------------------------------
Arrecife                       |  Lanzarote                 |       64497 |     2630.77 |    20 |   23.9 |   18.1 |      13 |     6.4
Puerto del Rosario             |  Fuerteventura             |       43390 |      144.94 |    16 |   23.2 |   17.8 |      12 |     5.6
Puntallana                     |  La Palma                  |        2549 |        69.2 |   420 |   23.1 |   19.8 |      10 |    12.4
Tías                           |  Lanzarote                 |       20801 |      308.99 |   200 |   22.8 |   16.9 |      13 |     6.4
Teguise                        |  Lanzarote                 |       23044 |       83.25 |   360 |   22.6 |   16.8 |      13 |     6.7
Haría                          |  Lanzarote             

In [1]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']

pipeline = [
    # 1. Sicherstellen, dass die Felder existieren
    {
        "$match": {
            "Isla": "Fuerteventura",
            "Temperature high": { "$exists": True },
            "Temperature low": { "$exists": True },
            "rainfall mm": {"$exists": True}
        }
    },

    # 2. Felder in Arrays transformieren
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "Población": 1,
            "Densidad": 1,
            "Altitud": 1,
            "high_arr": { "$objectToArray": "$Temperature high" },
            "low_arr": { "$objectToArray": "$Temperature low" },
            "rain_arr": { "$objectToArray": { "$ifNull": ["$rainfall mm", {}] } }
            
        }
    },

    # 3. Berechnungen (Durchschnitte, Min, Max)
    {
        "$addFields": {
            "avg_high": { "$avg": "$high_arr.v" },
            "avg_low": { "$avg": "$low_arr.v" },
            "max_temp": { "$max": "$high_arr.v" },
            "min_temp": { "$min": "$low_arr.v" },
            "avg_rain": { "$avg": "$rain_arr.v" },
            "water_arr": { "$objectToArray": { "$ifNull": ["$water", {}] } }
        }
    },

    # 4. Spanne und Wasser-Durchschnitt
    {
        "$addFields": {
            "avg_water": { "$avg": "$water_arr.v" },
            "amplitude": { "$subtract": ["$max_temp", "$min_temp"] }
        }
    },
    { "$sort": { "avg_high": -1 } }
]

results = collection.aggregate(pipeline)

header = f"{'Ciudad':<30} | {'Provincia':<25}  | {'Población':<10} | {'Densidad': <11} | {'Altitud':<5} |  {'High':<6} | {'Low':<6} | {'Spanne':<7} | {'Regen':<7} "
print(header)
print("-" * len(header))

for res in results:
    ciudad = res.get('Ciudad', 'Unbekannt')
    Provincia = res.get('Provincia', 'Unbekannt')
    Población = res.get('Población', 'Unbekannt')
    Densidad =  res.get('Densidad', 'Unbekannt')
    Altitud =  res.get('Altitud', 'Unbekannt')
    
    # Sicherer Fallback: Wenn ein Wert None ist, wird 0 verwendet
    a_high = round(res.get('avg_high') or 0, 1)
    a_low  = round(res.get('avg_low') or 0, 1)
    amp    = round(res.get('amplitude') or 0, 1)
    rain   = round(res.get('avg_rain') or 0, 1)
    
    w_val  = res.get('avg_water')
    water_str = f"{round(w_val, 1):>6}" if w_val is not None else "  ---  "

    print(f"{ciudad:<30} |  {Provincia:<25} |  {Población:>10} | {Densidad:>11} | {Altitud:>5} | {a_high:>6} | {a_low:>6} | {amp:>7} | {rain:>7}")

Ciudad                         | Provincia                  | Población  | Densidad    | Altitud |  High   | Low    | Spanne  | Regen   
----------------------------------------------------------------------------------------------------------------------------------------
Puerto del Rosario             |  Las Palmas                |       43390 |      144.94 |    16 |   23.2 |   17.8 |      12 |     5.6
La Oliva                       |  Las Palmas                |       27768 |       71.18 |   219 |   22.1 |   17.1 |      11 |     5.5
Tuíneje                        |  Las Palmas                |       15549 |       51.83 |   205 |   22.1 |   17.1 |      11 |     5.5
Pájara                         |  Las Palmas                |       21014 |       51.69 |   196 |   21.8 |   16.8 |      11 |     5.5


In [2]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']

pipeline = [
    # 1. Sicherstellen, dass die Felder existieren
    {
        "$match": {
            "Isla": "Lanzarote",
            "Temperature high": { "$exists": True },
            "Temperature low": { "$exists": True },
            "rainfall mm": {"$exists": True}
        }
    },

    # 2. Felder in Arrays transformieren
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "Población": 1,
            "Densidad": 1,
            "Altitud": 1,
            "high_arr": { "$objectToArray": "$Temperature high" },
            "low_arr": { "$objectToArray": "$Temperature low" },
            "rain_arr": { "$objectToArray": { "$ifNull": ["$rainfall mm", {}] } }
            
        }
    },

    # 3. Berechnungen (Durchschnitte, Min, Max)
    {
        "$addFields": {
            "avg_high": { "$avg": "$high_arr.v" },
            "avg_low": { "$avg": "$low_arr.v" },
            "max_temp": { "$max": "$high_arr.v" },
            "min_temp": { "$min": "$low_arr.v" },
            "avg_rain": { "$avg": "$rain_arr.v" },
            "water_arr": { "$objectToArray": { "$ifNull": ["$water", {}] } }
        }
    },

    # 4. Spanne und Wasser-Durchschnitt
    {
        "$addFields": {
            "avg_water": { "$avg": "$water_arr.v" },
            "amplitude": { "$subtract": ["$max_temp", "$min_temp"] }
        }
    },
    { "$sort": { "avg_high": -1 } }
]

results = collection.aggregate(pipeline)

header = f"{'Ciudad':<30} | {'Provincia':<25}  | {'Población':<10} | {'Densidad': <11} | {'Altitud':<5} |  {'High':<6} | {'Low':<6} | {'Spanne':<7} | {'Regen':<7} "
print(header)
print("-" * len(header))

for res in results:
    ciudad = res.get('Ciudad', 'Unbekannt')
    Provincia = res.get('Provincia', 'Unbekannt')
    Población = res.get('Población', 'Unbekannt')
    Densidad =  res.get('Densidad', 'Unbekannt')
    Altitud =  res.get('Altitud', 'Unbekannt')
    
    # Sicherer Fallback: Wenn ein Wert None ist, wird 0 verwendet
    a_high = round(res.get('avg_high') or 0, 1)
    a_low  = round(res.get('avg_low') or 0, 1)
    amp    = round(res.get('amplitude') or 0, 1)
    rain   = round(res.get('avg_rain') or 0, 1)
    
    w_val  = res.get('avg_water')
    water_str = f"{round(w_val, 1):>6}" if w_val is not None else "  ---  "

    print(f"{ciudad:<30} |  {Provincia:<25} |  {Población:>10} | {Densidad:>11} | {Altitud:>5} | {a_high:>6} | {a_low:>6} | {amp:>7} | {rain:>7}")

Ciudad                         | Provincia                  | Población  | Densidad    | Altitud |  High   | Low    | Spanne  | Regen   
----------------------------------------------------------------------------------------------------------------------------------------
Arrecife                       |  Las Palmas                |       64497 |     2630.77 |    20 |   23.9 |   18.1 |      13 |     6.4
Tías                           |  Las Palmas                |       20801 |      308.99 |   200 |   22.8 |   16.9 |      13 |     6.4
Teguise                        |  Las Palmas                |       23044 |       83.25 |   360 |   22.6 |   16.8 |      13 |     6.7
Haría                          |  Las Palmas                |        5365 |       45.58 |   270 |   22.3 |   16.8 |      12 |     7.1


In [3]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']

pipeline = [
    # 1. Sicherstellen, dass die Felder existieren
    {
        "$match": {
            "Isla": "La Palma",
            "Temperature high": { "$exists": True },
            "Temperature low": { "$exists": True },
            "rainfall mm": {"$exists": True}
        }
    },

    # 2. Felder in Arrays transformieren
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "Población": 1,
            "Densidad": 1,
            "Altitud": 1,
            "high_arr": { "$objectToArray": "$Temperature high" },
            "low_arr": { "$objectToArray": "$Temperature low" },
            "rain_arr": { "$objectToArray": { "$ifNull": ["$rainfall mm", {}] } }
            
        }
    },

    # 3. Berechnungen (Durchschnitte, Min, Max)
    {
        "$addFields": {
            "avg_high": { "$avg": "$high_arr.v" },
            "avg_low": { "$avg": "$low_arr.v" },
            "max_temp": { "$max": "$high_arr.v" },
            "min_temp": { "$min": "$low_arr.v" },
            "avg_rain": { "$avg": "$rain_arr.v" },
            "water_arr": { "$objectToArray": { "$ifNull": ["$water", {}] } }
        }
    },

    # 4. Spanne und Wasser-Durchschnitt
    {
        "$addFields": {
            "avg_water": { "$avg": "$water_arr.v" },
            "amplitude": { "$subtract": ["$max_temp", "$min_temp"] }
        }
    },
    { "$sort": { "avg_high": -1 } }
]

results = collection.aggregate(pipeline)

header = f"{'Ciudad':<30} | {'Provincia':<25}  | {'Población':<10} | {'Densidad': <11} | {'Altitud':<5} |  {'High':<6} | {'Low':<6} | {'Spanne':<7} | {'Regen':<7} "
print(header)
print("-" * len(header))

for res in results:
    ciudad = res.get('Ciudad', 'Unbekannt')
    Provincia = res.get('Provincia', 'Unbekannt')
    Población = res.get('Población', 'Unbekannt')
    Densidad =  res.get('Densidad', 'Unbekannt')
    Altitud =  res.get('Altitud', 'Unbekannt')
    
    # Sicherer Fallback: Wenn ein Wert None ist, wird 0 verwendet
    a_high = round(res.get('avg_high') or 0, 1)
    a_low  = round(res.get('avg_low') or 0, 1)
    amp    = round(res.get('amplitude') or 0, 1)
    rain   = round(res.get('avg_rain') or 0, 1)
    
    w_val  = res.get('avg_water')
    water_str = f"{round(w_val, 1):>6}" if w_val is not None else "  ---  "

    print(f"{ciudad:<30} |  {Provincia:<25} |  {Población:>10} | {Densidad:>11} | {Altitud:>5} | {a_high:>6} | {a_low:>6} | {amp:>7} | {rain:>7}")

Ciudad                         | Provincia                  | Población  | Densidad    | Altitud |  High   | Low    | Spanne  | Regen   
----------------------------------------------------------------------------------------------------------------------------------------
Puntallana                     |  Santa Cruz de Tenerife    |        2549 |        69.2 |   420 |   23.1 |   19.8 |      10 |    12.4
Santa Cruz de La Palma         |  Santa Cruz de Tenerife    |       15446 |      359.17 |     4 |   22.2 |   17.7 |      10 |    12.0
Los Llanos de Aridane          |  Santa Cruz de Tenerife    |       20648 |       561.8 |   340 |   20.8 |   15.8 |      11 |    11.7
El Paso                        |  Santa Cruz de Tenerife    |        7745 |       54.91 |   630 |   18.8 |   13.8 |      11 |    11.7


In [4]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']

pipeline = [
    # 1. Sicherstellen, dass die Felder existieren
    {
        "$match": {
            "Isla": "La Gomera",
            "Temperature high": { "$exists": True },
            "Temperature low": { "$exists": True },
            "rainfall mm": {"$exists": True}
        }
    },

    # 2. Felder in Arrays transformieren
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "Población": 1,
            "Densidad": 1,
            "Altitud": 1,
            "high_arr": { "$objectToArray": "$Temperature high" },
            "low_arr": { "$objectToArray": "$Temperature low" },
            "rain_arr": { "$objectToArray": { "$ifNull": ["$rainfall mm", {}] } }
            
        }
    },

    # 3. Berechnungen (Durchschnitte, Min, Max)
    {
        "$addFields": {
            "avg_high": { "$avg": "$high_arr.v" },
            "avg_low": { "$avg": "$low_arr.v" },
            "max_temp": { "$max": "$high_arr.v" },
            "min_temp": { "$min": "$low_arr.v" },
            "avg_rain": { "$avg": "$rain_arr.v" },
            "water_arr": { "$objectToArray": { "$ifNull": ["$water", {}] } }
        }
    },

    # 4. Spanne und Wasser-Durchschnitt
    {
        "$addFields": {
            "avg_water": { "$avg": "$water_arr.v" },
            "amplitude": { "$subtract": ["$max_temp", "$min_temp"] }
        }
    },
    { "$sort": { "avg_high": -1 } }
]

results = collection.aggregate(pipeline)

header = f"{'Ciudad':<30} | {'Provincia':<25}  | {'Población':<10} | {'Densidad': <11} | {'Altitud':<5} |  {'High':<6} | {'Low':<6} | {'Spanne':<7} | {'Regen':<7} "
print(header)
print("-" * len(header))

for res in results:
    ciudad = res.get('Ciudad', 'Unbekannt')
    Provincia = res.get('Provincia', 'Unbekannt')
    Población = res.get('Población', 'Unbekannt')
    Densidad =  res.get('Densidad', 'Unbekannt')
    Altitud =  res.get('Altitud', 'Unbekannt')
    
    # Sicherer Fallback: Wenn ein Wert None ist, wird 0 verwendet
    a_high = round(res.get('avg_high') or 0, 1)
    a_low  = round(res.get('avg_low') or 0, 1)
    amp    = round(res.get('amplitude') or 0, 1)
    rain   = round(res.get('avg_rain') or 0, 1)
    
    w_val  = res.get('avg_water')
    water_str = f"{round(w_val, 1):>6}" if w_val is not None else "  ---  "

    print(f"{ciudad:<30} |  {Provincia:<25} |  {Población:>10} | {Densidad:>11} | {Altitud:>5} | {a_high:>6} | {a_low:>6} | {amp:>7} | {rain:>7}")

Ciudad                         | Provincia                  | Población  | Densidad    | Altitud |  High   | Low    | Spanne  | Regen   
----------------------------------------------------------------------------------------------------------------------------------------


In [5]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']

pipeline = [
    # 1. Sicherstellen, dass die Felder existieren
    {
        "$match": {
            "Isla": "Tenerife",
            "Temperature high": { "$exists": True },
            "Temperature low": { "$exists": True },
            "rainfall mm": {"$exists": True}
        }
    },

    # 2. Felder in Arrays transformieren
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "Población": 1,
            "Densidad": 1,
            "Altitud": 1,
            "high_arr": { "$objectToArray": "$Temperature high" },
            "low_arr": { "$objectToArray": "$Temperature low" },
            "rain_arr": { "$objectToArray": { "$ifNull": ["$rainfall mm", {}] } }
            
        }
    },

    # 3. Berechnungen (Durchschnitte, Min, Max)
    {
        "$addFields": {
            "avg_high": { "$avg": "$high_arr.v" },
            "avg_low": { "$avg": "$low_arr.v" },
            "max_temp": { "$max": "$high_arr.v" },
            "min_temp": { "$min": "$low_arr.v" },
            "avg_rain": { "$avg": "$rain_arr.v" },
            "water_arr": { "$objectToArray": { "$ifNull": ["$water", {}] } }
        }
    },

    # 4. Spanne und Wasser-Durchschnitt
    {
        "$addFields": {
            "avg_water": { "$avg": "$water_arr.v" },
            "amplitude": { "$subtract": ["$max_temp", "$min_temp"] }
        }
    },
    { "$sort": { "avg_high": -1 } }
]

results = collection.aggregate(pipeline)

header = f"{'Ciudad':<30} | {'Provincia':<25}  | {'Población':<10} | {'Densidad': <11} | {'Altitud':<5} |  {'High':<6} | {'Low':<6} | {'Spanne':<7} | {'Regen':<7} "
print(header)
print("-" * len(header))

for res in results:
    ciudad = res.get('Ciudad', 'Unbekannt')
    Provincia = res.get('Provincia', 'Unbekannt')
    Población = res.get('Población', 'Unbekannt')
    Densidad =  res.get('Densidad', 'Unbekannt')
    Altitud =  res.get('Altitud', 'Unbekannt')
    
    # Sicherer Fallback: Wenn ein Wert None ist, wird 0 verwendet
    a_high = round(res.get('avg_high') or 0, 1)
    a_low  = round(res.get('avg_low') or 0, 1)
    amp    = round(res.get('amplitude') or 0, 1)
    rain   = round(res.get('avg_rain') or 0, 1)
    
    w_val  = res.get('avg_water')
    water_str = f"{round(w_val, 1):>6}" if w_val is not None else "  ---  "

    print(f"{ciudad:<30} |  {Provincia:<25} |  {Población:>10} | {Densidad:>11} | {Altitud:>5} | {a_high:>6} | {a_low:>6} | {amp:>7} | {rain:>7}")

Ciudad                         | Provincia                  | Población  | Densidad    | Altitud |  High   | Low    | Spanne  | Regen   
----------------------------------------------------------------------------------------------------------------------------------------


In [6]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']

pipeline = [
    # 1. Sicherstellen, dass die Felder existieren
    {
        "$match": {
            "Isla": "El Hierro",
            "Temperature high": { "$exists": True },
            "Temperature low": { "$exists": True },
            "rainfall mm": {"$exists": True}
        }
    },

    # 2. Felder in Arrays transformieren
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "Población": 1,
            "Densidad": 1,
            "Altitud": 1,
            "high_arr": { "$objectToArray": "$Temperature high" },
            "low_arr": { "$objectToArray": "$Temperature low" },
            "rain_arr": { "$objectToArray": { "$ifNull": ["$rainfall mm", {}] } }
            
        }
    },

    # 3. Berechnungen (Durchschnitte, Min, Max)
    {
        "$addFields": {
            "avg_high": { "$avg": "$high_arr.v" },
            "avg_low": { "$avg": "$low_arr.v" },
            "max_temp": { "$max": "$high_arr.v" },
            "min_temp": { "$min": "$low_arr.v" },
            "avg_rain": { "$avg": "$rain_arr.v" },
            "water_arr": { "$objectToArray": { "$ifNull": ["$water", {}] } }
        }
    },

    # 4. Spanne und Wasser-Durchschnitt
    {
        "$addFields": {
            "avg_water": { "$avg": "$water_arr.v" },
            "amplitude": { "$subtract": ["$max_temp", "$min_temp"] }
        }
    },
    { "$sort": { "avg_high": -1 } }
]

results = collection.aggregate(pipeline)

header = f"{'Ciudad':<30} | {'Provincia':<25}  | {'Población':<10} | {'Densidad': <11} | {'Altitud':<5} |  {'High':<6} | {'Low':<6} | {'Spanne':<7} | {'Regen':<7} "
print(header)
print("-" * len(header))

for res in results:
    ciudad = res.get('Ciudad', 'Unbekannt')
    Provincia = res.get('Provincia', 'Unbekannt')
    Población = res.get('Población', 'Unbekannt')
    Densidad =  res.get('Densidad', 'Unbekannt')
    Altitud =  res.get('Altitud', 'Unbekannt')
    
    # Sicherer Fallback: Wenn ein Wert None ist, wird 0 verwendet
    a_high = round(res.get('avg_high') or 0, 1)
    a_low  = round(res.get('avg_low') or 0, 1)
    amp    = round(res.get('amplitude') or 0, 1)
    rain   = round(res.get('avg_rain') or 0, 1)
    
    w_val  = res.get('avg_water')
    water_str = f"{round(w_val, 1):>6}" if w_val is not None else "  ---  "

    print(f"{ciudad:<30} |  {Provincia:<25} |  {Población:>10} | {Densidad:>11} | {Altitud:>5} | {a_high:>6} | {a_low:>6} | {amp:>7} | {rain:>7}")

Ciudad                         | Provincia                  | Población  | Densidad    | Altitud |  High   | Low    | Spanne  | Regen   
----------------------------------------------------------------------------------------------------------------------------------------


In [7]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']

pipeline = [
    # 1. Sicherstellen, dass die Felder existieren
    {
        "$match": {
            "Isla": "Gran Canaria",
            "Temperature high": { "$exists": True },
            "Temperature low": { "$exists": True },
            "rainfall mm": {"$exists": True}
        }
    },

    # 2. Felder in Arrays transformieren
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "Población": 1,
            "Densidad": 1,
            "Altitud": 1,
            "high_arr": { "$objectToArray": "$Temperature high" },
            "low_arr": { "$objectToArray": "$Temperature low" },
            "rain_arr": { "$objectToArray": { "$ifNull": ["$rainfall mm", {}] } }
            
        }
    },

    # 3. Berechnungen (Durchschnitte, Min, Max)
    {
        "$addFields": {
            "avg_high": { "$avg": "$high_arr.v" },
            "avg_low": { "$avg": "$low_arr.v" },
            "max_temp": { "$max": "$high_arr.v" },
            "min_temp": { "$min": "$low_arr.v" },
            "avg_rain": { "$avg": "$rain_arr.v" },
            "water_arr": { "$objectToArray": { "$ifNull": ["$water", {}] } }
        }
    },

    # 4. Spanne und Wasser-Durchschnitt
    {
        "$addFields": {
            "avg_water": { "$avg": "$water_arr.v" },
            "amplitude": { "$subtract": ["$max_temp", "$min_temp"] }
        }
    },
    { "$sort": { "avg_high": -1 } }
]

results = collection.aggregate(pipeline)

header = f"{'Ciudad':<30} | {'Provincia':<25}  | {'Población':<10} | {'Densidad': <11} | {'Altitud':<5} |  {'High':<6} | {'Low':<6} | {'Spanne':<7} | {'Regen':<7} "
print(header)
print("-" * len(header))

for res in results:
    ciudad = res.get('Ciudad', 'Unbekannt')
    Provincia = res.get('Provincia', 'Unbekannt')
    Población = res.get('Población', 'Unbekannt')
    Densidad =  res.get('Densidad', 'Unbekannt')
    Altitud =  res.get('Altitud', 'Unbekannt')
    
    # Sicherer Fallback: Wenn ein Wert None ist, wird 0 verwendet
    a_high = round(res.get('avg_high') or 0, 1)
    a_low  = round(res.get('avg_low') or 0, 1)
    amp    = round(res.get('amplitude') or 0, 1)
    rain   = round(res.get('avg_rain') or 0, 1)
    
    w_val  = res.get('avg_water')
    water_str = f"{round(w_val, 1):>6}" if w_val is not None else "  ---  "

    print(f"{ciudad:<30} |  {Provincia:<25} |  {Población:>10} | {Densidad:>11} | {Altitud:>5} | {a_high:>6} | {a_low:>6} | {amp:>7} | {rain:>7}")

Ciudad                         | Provincia                  | Población  | Densidad    | Altitud |  High   | Low    | Spanne  | Regen   
----------------------------------------------------------------------------------------------------------------------------------------
